# Notebook 3 - Train, Validation & Test

Topics covered:
- Training Dataset
- Validation Dataset
- Test Dataset
- Train/Test Split
- Train/Validation/Test Split
- Cross Validation
- K-Fold Cross Validation
- Stratified K-Fold
- Random State
- Data Leakage
- Overfitting
- Underfitting

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Load Dataset

We will use the Breast Cancer dataset from scikit-learn.

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)

print("\nFirst 5 rows:")
print(X.head())

print("\nTarget distribution:")
print(y.value_counts())

## 3. Training, Validation and Test Dataset

**Training Dataset:** Used to train the model.

**Validation Dataset:** Used to tune the model and select the best model.

**Test Dataset:** Used only once at the end for final evaluation.

## 4. Train/Test Split

Here we divide the dataset into:
- 80% Training
- 20% Testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 5. Train/Validation/Test Split

We will create:
- 60% Training
- 20% Validation
- 20% Testing

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nApproximate percentages:")
print("Training:", round(len(X_train) / len(X) * 100, 2), "%")
print("Validation:", round(len(X_val) / len(X) * 100, 2), "%")
print("Testing:", round(len(X_test) / len(X) * 100, 2), "%")

## 6. Standardization Without Data Leakage

The scaler must be fitted only on the training dataset.

Then the same scaler is used to transform validation and test data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:", X_train_scaled.shape)
print("Validation data scaled:", X_val_scaled.shape)
print("Test data scaled:", X_test_scaled.shape)

## 7. Train the Model

We train Logistic Regression using only the training dataset.

In [ ]:
model = LogisticRegression(max_iter=5000, random_state=42)

model.fit(X_train_scaled, y_train)

print("Model training completed.")

## 8. Validation Dataset

The validation dataset is used to check model performance while developing and tuning the model.

In [ ]:
y_val_pred = model.predict(X_val_scaled)

validation_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation Accuracy:", round(validation_accuracy, 4))

## 9. Test Dataset

The test dataset should not be used for model tuning.

It should be used only for the final unbiased evaluation.

In [ ]:
y_test_pred = model.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_test_pred)

print("Test Accuracy:", round(test_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=data.target_names))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

## 10. Why Test Data Should Not Be Used for Model Tuning

The test dataset must remain unseen during model development.

If we repeatedly check test performance and change the model based on those results, the model-selection process starts learning from the test data.

This can produce an overly optimistic test score and reduce the reliability of the final evaluation.

Correct workflow:

Dataset → Training → Validation/Tuning → Final Model → Test Evaluation

## 11. K-Fold Cross Validation

K-Fold Cross Validation divides the training data into K parts called folds.

The model is trained K times. Each fold is used once as validation data while the remaining folds are used for training.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring="accuracy"
)

print("K-Fold Scores:", cv_scores)
print("Mean K-Fold Accuracy:", round(cv_scores.mean(), 4))

## 12. Stratified K-Fold Cross Validation

Stratified K-Fold keeps the class distribution approximately the same in every fold.

It is especially useful for classification problems.

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stratified_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring="accuracy"
)

print("Stratified K-Fold Scores:", stratified_scores)
print("Mean Stratified K-Fold Accuracy:", round(stratified_scores.mean(), 4))

## 13. Random State

`random_state` controls the randomness used during operations such as data splitting and shuffling.

Using the same random_state produces reproducible results.

In [ ]:
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Same training split:", X_train_1.equals(X_train_2))

## 14. Data Leakage

Data leakage happens when information from outside the training data is used during model training.

Example of incorrect scaling:

1. Combine training and test data.
2. Fit the scaler on the complete dataset.
3. Split the data.

This allows information from the test set to influence the transformation.

Correct approach:

1. Split the data first.
2. Fit preprocessing only on training data.
3. Transform validation and test data using the fitted preprocessing object.

## 15. Overfitting

Overfitting happens when a model learns the training data too closely, including noise.

Typical pattern:
- Very high training accuracy
- Lower validation accuracy

In [ ]:
complex_tree = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

complex_tree.fit(X_train, y_train)

train_pred = complex_tree.predict(X_train)
val_pred = complex_tree.predict(X_val)

train_accuracy = accuracy_score(y_train, train_pred)
val_accuracy = accuracy_score(y_val, val_pred)

print("Training Accuracy:", round(train_accuracy, 4))
print("Validation Accuracy:", round(val_accuracy, 4))

## 16. Underfitting

Underfitting happens when the model is too simple to learn the important patterns in the data.

Typical pattern:
- Low training accuracy
- Low validation accuracy

In [ ]:
simple_tree = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

simple_tree.fit(X_train, y_train)

simple_train_pred = simple_tree.predict(X_train)
simple_val_pred = simple_tree.predict(X_val)

simple_train_accuracy = accuracy_score(y_train, simple_train_pred)
simple_val_accuracy = accuracy_score(y_val, simple_val_pred)

print("Training Accuracy:", round(simple_train_accuracy, 4))
print("Validation Accuracy:", round(simple_val_accuracy, 4))

## 17. Underfitting vs Good Fit vs Overfitting

| Situation | Training Performance | Validation Performance |
|---|---|---|
| Underfitting | Low | Low |
| Good Fit | High | High and similar to training |
| Overfitting | Very High | Much lower than training |

## 18. Final Summary

- Training data is used to learn model parameters.
- Validation data is used for model selection and tuning.
- Test data is reserved for final evaluation.
- Train/Test Split divides data into training and testing sets.
- Train/Validation/Test Split creates three separate datasets.
- Cross Validation gives a more reliable estimate of model performance.
- K-Fold divides data into K folds.
- Stratified K-Fold preserves class proportions.
- Random State makes random operations reproducible.
- Data Leakage occurs when information improperly flows into training.
- Overfitting means the model learns training data too closely.
- Underfitting means the model is too simple to learn important patterns.

### Correct ML Workflow

Dataset → Train/Validation/Test Split → Preprocessing → Model Training → Validation/Tuning → Final Model → Test Evaluation